# MagmaBallz — Kaggle Marathon: solverV3 vs structural_cache

> **Target:** `benchmarks/marathon_hard100.json` v2 manifest (100 problems: 20×extra_hard, 80×order5, seed 20260901, 300s global budget, 0 LLM tokens) comparing 3 runs:
> - `solverV3` — marathon harness, global 300s budget (current best marathon solver)
> - `structural_cache_marathon` — the marathon port of the SOLO structural-cache solver, same harness/budget
> - `structural_cache` (SOLO harness) — per-problem 300s **and** a compressed 6s variant, to measure how the SOLO solver degrades under marathon-style per-problem pressure

Question this notebook answers: **is the structural-cache SOLO solver suitable for the marathon track right now?** Local reference results (doc `docs/marathon_hard100_benchmark.md`): V3 90/100 in 147.1s, port 90/100 in 296.6s, complementary accepted sets (merging = 98/100). This notebook re-runs everything on Kaggle and adds the SOLO-side measurements.

Fully self-contained: installs Lean/lake/Mathlib, builds the judge, runs all three harnesses with resume, aggregates. Enable **Internet** in Kaggle settings. Expected runtime: ~30 min setup + ~10 min marathon runs + ~2h SOLO 300s + ~40 min SOLO 6s (workers=4). Kaggle limit 12h.


## 0 — Kaggle checklist

- Settings → **Internet ON** (needed for `elan`, `lake exe cache get`, `pip install openai`)
- Settings → **Persist outputs** ON if you want results under `/kaggle/working`
- Secrets → add `OPENROUTER_API_KEY` or `OPENAI_API_KEY` if you want LLM calls (solvers fall back to deterministic search otherwise)
- Accelerator: **None/CPU** is fine (Lean is CPU-bound, 4 cores → `workers=4`)


In [ ]:
# ── 0.1 Detect Kaggle env + set paths (robust to papermill cwd) ──
import os, sys, pathlib, json, subprocess, shutil, textwrap
from pathlib import Path

def find_repo_root() -> Path:
    # 1. Check cwd and parents for repo
    cwd = Path.cwd().resolve()
    for cand in [cwd] + list(cwd.parents)[:6]:
        if (cand / "lean-toolchain").exists() and (cand / "lakefile.lean").exists():
            return cand
        if (cand / "MagmaBallz" / "lean-toolchain").exists():
            return (cand / "MagmaBallz").resolve()
    # 2. Check known locations
    for base in [Path("/home/anhtu77/Coding/MagmaBallz"), Path("/kaggle/working/MagmaBallz"), Path("/tmp/MagmaBallz"), Path("/tmp/MagmaBallz_clone")]:
        if (base / "lean-toolchain").exists() and (base / "lakefile.lean").exists():
            return base.resolve()
    # 3. Brute-force find under /kaggle and /home (limited)
    for search_root in ["/kaggle", "/home"]:
        try:
            out = subprocess.check_output(["find", str(search_root), "-maxdepth", "5", "-name", "lean-toolchain", "-type", "f"], text=True, timeout=5)
            for line in out.splitlines():
                cand = Path(line.strip()).parent.resolve()
                if (cand / "lakefile.lean").exists():
                    return cand
        except: pass
    # 4. Fallback: if on Kaggle, working dir is intended clone parent, not repo itself
    if Path("/kaggle").exists():
        # return a path that will be used as clone target, not the empty working dir
        return Path("/kaggle/working").resolve()
    return cwd

IS_KAGGLE = Path("/kaggle").exists()
print(f"IS_KAGGLE={IS_KAGGLE}")
print(f"python={sys.version}")
print(f"initial cwd={Path.cwd()}")
try:
    print(f"initial ls={list(Path.cwd().iterdir())[:15]}")
except: print("ls failed")

REPO_ROOT = find_repo_root()
print(f"REPO_ROOT found: {REPO_ROOT}  exists lean-toolchain={(REPO_ROOT / 'lean-toolchain').exists()}")
# Only chdir if REPO_ROOT actually contains repo
if (REPO_ROOT / "lean-toolchain").exists():
    try:
        os.chdir(REPO_ROOT)
        print(f"chdir to {REPO_ROOT} -> cwd={Path.cwd()}")
        get_ipython().run_line_magic("cd", str(REPO_ROOT))
    except Exception as e:
        print(f"chdir failed: {e}")
else:
    print(f"REPO_ROOT does not contain repo yet, staying in {Path.cwd()} - will clone in next cell")

print(f"REPO_ROOT={REPO_ROOT}")
print(f"cwd after={Path.cwd()}")


In [ ]:
# ── 0.2 System check + Python deps ──
!python3 --version
!pip --version
!df -h | head -20
!nproc; free -h | head -5

# Install Python deps (idempotent)
!pip install -q openai tqdm pandas matplotlib
import openai, tqdm, pandas
print(f"openai={openai.__version__}")

## 1 — Lean toolchain setup (full, Kaggle-safe, idempotent)

Mirrors `scripts/setup.sh:1` — elan → toolchain → lake update → cache get → build judge modules. Re-running is safe (skips installed parts).

In [ ]:
# ── 1.1 Install elan if missing ──
import shutil, subprocess, os
from pathlib import Path
!which elan || echo "elan not found"
!elan --version 2>&1 | head -5

if not shutil.which("elan"):
    print("Installing elan...")
    !curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain none
    # Update PATH for this notebook kernel
    os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ["PATH"]
    !export PATH="$HOME/.elan/bin:$PATH" && elan --version
else:
    print("elan already installed")
    os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ.get("PATH","")
    !elan --version
    !which lean; which lake || true

In [ ]:
# ── 1.2 Install pinned toolchain + set default (REPO_ROOT-aware) ──
from pathlib import Path
import os
# Ensure in REPO_ROOT
os.chdir(REPO_ROOT)
try: get_ipython().run_line_magic("cd", str(REPO_ROOT))
except: pass

toolchain_path = REPO_ROOT / "lean-toolchain"
print(f"toolchain_path={toolchain_path} exists={toolchain_path.exists()} cwd={Path.cwd()}")
toolchain = toolchain_path.read_text().strip()
print(f"Required toolchain: {toolchain}")
get_ipython().system('elan toolchain list 2>&1 | head -20')
# Use shell with REPO_ROOT env
get_ipython().system('elan toolchain install "$toolchain" 2>&1 | tail -20')
get_ipython().system('elan default "$toolchain"')
get_ipython().system('lean --version')
get_ipython().system('lake --version')


In [ ]:
# ── 1.3 Fetch Mathlib + build judge (heavy, cached) ──
# This is the slow part (~2GB cache, 5-10min with cache, 1h+ without).
import os
from pathlib import Path
os.chdir(REPO_ROOT)
try: get_ipython().run_line_magic("cd", str(REPO_ROOT))
except: pass
os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ["PATH"]

# Show lake manifest pinned rev
get_ipython().system('cat lake-manifest.json | head -30')
print("Running lake update...")
get_ipython().system('lake update 2>&1 | tail -30')

print("Downloading Mathlib cache...")
get_ipython().system('lake exe cache get 2>&1 | tail -30')

print("Building judge modules (JudgeMagma, JudgeDecide, JudgeFinOp, JudgeSupport)...")
get_ipython().system('lake build JudgeMagma.Magma JudgeDecide.DecideBang JudgeFinOp.MemoFinOp JudgeSupport.Inspect 2>&1 | tail -50')
print("Lake build done")
get_ipython().system('ls -lh .lake/build/lib/Judge* 2>&1 | head -20')


In [ ]:
# ── 1.4 Write .env.judge + export to notebook env (REPO_ROOT-aware) ──
import subprocess, pathlib, os, shutil
from pathlib import Path
os.chdir(REPO_ROOT)
try: get_ipython().run_line_magic("cd", str(REPO_ROOT))
except: pass
# Find lean/lake
lean_bin = shutil.which("lean") or str(Path.home() / ".elan" / "bin" / "lean")
lake_bin = shutil.which("lake") or str(Path.home() / ".elan" / "bin" / "lake")
get_ipython().system('which lean')
get_ipython().system('which lake')
lean_bin = shutil.which("lean")
lake_bin = shutil.which("lake")
print(f"LEAN_BIN={lean_bin}")
print(f"LAKE_BIN={lake_bin}")

env_file = REPO_ROOT / ".env.judge"
env_file.write_text(f'''# Auto-generated by Kaggle notebook
export LEAN_BIN="{lean_bin}"
export LAKE_BIN="{lake_bin}"
export PATH="$HOME/.elan/bin:$PATH"
''')
print(env_file.read_text())
os.environ["LEAN_BIN"] = lean_bin
os.environ["LAKE_BIN"] = lake_bin
os.environ["PATH"] = str(Path.home() / ".elan" / "bin") + ":" + os.environ["PATH"]
print("Env exported")


In [ ]:
# ── 1.5 Smoke test judge (must be accepted) ──
import json, sys, os
from pathlib import Path
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
from judge.verify import verify_answer, JudgeConfig
lean_bin = Path(os.environ["LEAN_BIN"])
lake_bin = Path(os.environ["LAKE_BIN"])
config = JudgeConfig(lake_bin=lake_bin, lean_bin=lean_bin)
problem = json.loads((REPO_ROOT / "tests/fixtures/problems/p_true_basic.json").read_text())
answer = (REPO_ROOT / "tests/fixtures/answers/accepted_true_basic.answer.json").read_text()
result = verify_answer(problem, answer, config=config)
print(json.dumps({"status": result["status"]}, indent=2))
assert result["status"] == "accepted", f"Smoke failed: {result}"
print("✅ Judge smoke PASSED")

# Also run harness quick check (optional, ~30s)
get_ipython().system('python3 scripts/run_harness.py 2>&1 | tail -100')


## 3 — Profile, manifest, expected answers

The v2 manifest is label-free (answers stripped). Expected labels are recovered by mapping manifest ids back to the source JSONL rows (`examples/problems/evaluation_extra_hard.jsonl`, `evaluation_order5.jsonl`). These labels are used ONLY for analysis of the SOLO run (which needs a ground truth to score); the marathon runs are scored by the Lean judge alone.


In [ ]:
import json, hashlib, copy, platform, sys, time, os
from pathlib import Path
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
from pipeline.marathon_benchmark import load_profile, load_manifest, _fingerprint

PROFILE_PATH = REPO_ROOT / "benchmarks" / "marathon_hard100.json"
profile = load_profile(PROFILE_PATH)
manifest_path = (REPO_ROOT / profile["manifest"]).resolve()
problems = load_manifest(manifest_path)
print(f"profile: {profile['name']}  seed={profile['selection']['seed']}")
print(f"manifest: {manifest_path.relative_to(REPO_ROOT)}  rows={len(problems)}")
print(f"budget: {profile['budget_seconds']}s / {profile['budget_tokens']} tokens")
from collections import Counter
print(f"difficulties: {dict(Counter(p['difficulty'] for p in problems))}")

# expected answers: manifest id -> source row (source rows carry 'answer')
SRC_PATHS = {"extra_hard": "examples/problems/evaluation_extra_hard.jsonl",
             "order5_normal": "examples/problems/evaluation_order5.jsonl"}
source_by_id = {}
for path in SRC_PATHS.values():
    for line in (REPO_ROOT / path).open(encoding="utf-8"):
        row = json.loads(line)
        source_by_id[row["id"]] = row
expected = {p["id"]: source_by_id[p["id"]]["answer"] for p in problems}
labels = Counter(expected.values())
print(f"expected labels: {dict(labels)}")
print("all manifest ids found in source rows:",
      all(p["id"] in source_by_id for p in problems))


## 4 — Marathon runs (global 300s budget)

Both solvers run under `pipeline.marathon_runner.run_marathon` with the same 300s/0-token budget and are scored by `pipeline.marathon_score` (Lean judge). Resume: if a completed run with a matching fingerprint already exists under `pipeline/results/marathon_hard100/`, the summary is reused instead of re-running — so the notebook is safe to re-run.


In [ ]:
import shutil, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from pipeline.marathon_runner import run_marathon
from pipeline.marathon_score import score_marathon

MARATHON_CANDIDATES = {
    "solverV3": REPO_ROOT / "my_submission/marathon/solverV3.py",
    "structural_cache_marathon": REPO_ROOT / "my_submission/marathon/structural_cache_marathon.py",
}
OUT_ROOT = REPO_ROOT / profile["output_directory"]
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def load_marathon_run(name, source):
    fp = _fingerprint(PROFILE_PATH.resolve(), manifest_path, source)
    matches = sorted(OUT_ROOT.glob(f"*/{name}/summary.json"))
    for path in matches:
        summary = json.loads(path.read_text())
        if summary.get("fingerprint") == fp:
            return path.parent, summary, True
    return None, None, False

def run_marathon_candidate(name, source):
    run_dir, existing, reused = load_marathon_run(name, source)
    if reused:
        print(f"{name}: REUSED {run_dir}")
        return existing
    stamp = time.strftime("%Y%m%d_%H%M%S")
    run_dir = OUT_ROOT / stamp / name
    run_dir.mkdir(parents=True, exist_ok=True)
    output_path = run_dir / "answers.jsonl"
    log_path = run_dir / "run.log"
    with tempfile.TemporaryDirectory(prefix="magma-marathon-") as raw_stage:
        stage = Path(raw_stage)
        shutil.copy2(source, stage / "solver.py")
        with log_path.open("w", encoding="utf-8") as log:
            result = run_marathon(
                submission_dir=stage, manifest_path=manifest_path,
                output_path=output_path, scratch_dir=run_dir / "scratch",
                budget_seconds=float(profile["budget_seconds"]),
                budget_tokens=int(profile["budget_tokens"]),
                enable_proxy=False, log_stream=log)
            summary = score_marathon(
                manifest_problems=result.manifest_problems, output_path=output_path,
                wall_seconds=result.wall_seconds, sigterm_fired=result.sigterm_fired,
                sigkill_fired=result.sigkill_fired,
                tokens_used=result.tokens_used, tokens_exhausted=result.tokens_exhausted,
                log_stream=log).to_dict()
    payload = {"name": name, "source": str(source),
               "fingerprint": _fingerprint(PROFILE_PATH.resolve(), manifest_path, source),
               **summary}
    (run_dir / "summary.json").write_text(json.dumps(payload, indent=2))
    print(f"{name}: score={summary['score']}/100 wall={summary['wall_seconds']:.1f}s by_status={summary['by_status']}")
    return payload

for name, source in MARATHON_CANDIDATES.items():
    print(f"exists={source.exists()}")


In [ ]:
import os; os.chdir(REPO_ROOT); import sys; sys.path.insert(0, str(REPO_ROOT))
marathon_results = {}
for name, source in MARATHON_CANDIDATES.items():
    if not source.exists():
        print(f"SKIP {name}: missing {source}")
        continue
    print(f"\n### {name}")
    marathon_results[name] = run_marathon_candidate(name, source)
    if Path("/kaggle/working").exists():
        !cp -r pipeline/results/marathon_hard100 /kaggle/working/ 2>&1 | tail -2
print("\nMarathon scoreboard:")
for name, s in marathon_results.items():
    print(f"  {name:28s} {s['score']}/100  wall={s['wall_seconds']:.1f}s  {s['by_status']}")


## 5 — SOLO runs of structural_cache (per-problem budgets)

The SOLO harness (`pipeline.proxy.run_solver`) isolates each problem in its own process with a per-problem timeout — that is structural_cache's native environment. Two passes:

- **300s/problem** (SOLO rules, same as `benchmarks/solo_hard_100.json`) — measures its true ceiling per problem.
- **6s/problem** (compressed) — approximates marathon per-problem pressure (300s/100 problems); measures how much of its machinery survives a port.

No LLM key? The solver's LLM-collaboration passes fail immediately and it runs deterministic-only — same posture as the marathon 0-token runs. Resume is on.


In [ ]:
import copy, json, hashlib, shutil, tempfile, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
os.chdir(REPO_ROOT); sys.path.insert(0, str(REPO_ROOT))
from pipeline.proxy import load_config, run_solver

SOLO_SUBMISSION = REPO_ROOT / "my_submission/solo_variants/structural_cache.py"
BASE_CONFIG = load_config(REPO_ROOT / "pipeline/config.json")
KAGGLE_WORKERS = 4
SOLO_PASSES = [("solo300", 300), ("solo6", 6)]

# stage the single-file submission
staging = tempfile.TemporaryDirectory(prefix="magmaballz-solo-")
staged = Path(staging.name)
shutil.copyfile(SOLO_SUBMISSION, staged / "solver.py")

solo_out_root = REPO_ROOT / "pipeline/results/marathon_vs_solo"
solo_out_root.mkdir(parents=True, exist_ok=True)

def load_solo_done(run_dir):
    done = {}
    for f in run_dir.glob("*.results.jsonl"):
        for line in f.read_text(encoding="utf-8").splitlines():
            if line.strip():
                row = json.loads(line)
                done[row["id"]] = row
    return done

def run_solo_pass(name, timeout_seconds, workers=KAGGLE_WORKERS):
    run_dir = solo_out_root / name
    run_dir.mkdir(parents=True, exist_ok=True)
    done = load_solo_done(run_dir)
    pending = [p for p in problems if p["id"] not in done]
    print(f"\nSOLO pass {name}: timeout={timeout_seconds}s  resumed={len(done)}  pending={len(pending)}")
    def one(p):
        cfg = copy.deepcopy(BASE_CONFIG)
        cfg["solver"]["timeout_seconds"] = timeout_seconds
        public = dict(source_by_id[p["id"]])
        public.pop("answer", None)
        t0 = time.monotonic()
        res = run_solver(staged, public, cfg)
        row = {
            "id": p["id"], "difficulty": p["difficulty"],
            "expected_answer": expected[p["id"]],
            "elapsed_seconds": round(time.monotonic() - t0, 3),
            **res,
        }
        row["label_match"] = (res.get("verdict") == ("true" if expected[p["id"]] else "false")
                              if res.get("solved") else None)
        return row
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(one, p): p for p in pending}
        for fut in as_completed(futs):
            row = fut.result()
            with (run_dir / "all.results.jsonl").open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(row) + "\n")
            done[row["id"]] = row
    (run_dir / "summary.json").write_text(json.dumps({
        "name": name, "timeout_seconds": timeout_seconds,
        "solved": sum(1 for r in done.values() if r.get("solved")),
        "correct": sum(1 for r in done.values() if r.get("label_match")),
        "total": len(done),
        "elapsed_seconds": round(sum(r["elapsed_seconds"] for r in done.values()), 1),
    }, indent=2))
    n = len(done)
    solved = sum(1 for r in done.values() if r.get("solved"))
    correct = sum(1 for r in done.values() if r.get("label_match"))
    print(f"SOLO pass {name}: {correct}/{n} correct  {solved} solved  "
          f"sum(elapsed)={sum(r['elapsed_seconds'] for r in done.values()):.0f}s")
    return done


In [ ]:
import os; os.chdir(REPO_ROOT); import sys; sys.path.insert(0, str(REPO_ROOT))
solo_done = {}
for name, timeout in SOLO_PASSES:
    solo_done[name] = run_solo_pass(name, timeout)
print("\nAll SOLO passes complete")


## 6 — Analysis

Scoreboards, per-problem elapsed, overlap matrix, and a greedy global-budget simulation that predicts a naive per-problem port of the SOLO solver under a 300s global cap.


In [ ]:
import json, pathlib
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
os.chdir(REPO_ROOT)

def accepted_ids(summary):
    return {p["id"] for p in summary["per_problem"] if p["status"] == "accepted"}

rows = []
for name, s in marathon_results.items():
    acc = accepted_ids(s)
    rows.append({
        "run": name, "harness": "marathon", "budget": "global 300s",
        "accepted": s["score"], "wall_seconds": s["wall_seconds"],
        "false": sum(1 for i in acc if expected[i] is False),
        "true": sum(1 for i in acc if expected[i] is True),
        "accepted_ids": acc,
    })
for name, done in solo_done.items():
    acc = {i for i, r in done.items() if r.get("label_match")}
    rows.append({
        "run": f"structural_cache ({name})", "harness": "solo",
        "budget": f"{SOLO_PASSES[[n for n,_ in SOLO_PASSES].index(name)][1]}s/problem",
        "accepted": len(acc),
        "wall_seconds": sum(r["elapsed_seconds"] for r in done.values()),
        "false": sum(1 for i in acc if expected[i] is False),
        "true": sum(1 for i in acc if expected[i] is True),
        "accepted_ids": acc,
    })
df = pd.DataFrame(rows)
df["accepted"] = df["accepted"].astype(int)
df["false"] = df["false"].astype(int); df["true"] = df["true"].astype(int)
df = df.sort_values(["accepted", "wall_seconds"], ascending=[False, True])
display(df.drop(columns=["accepted_ids"]))
print("accepted == judge-accepted (marathon) or label-correct solved (solo); wall = solver wall (marathon) / sum of per-problem elapsed (solo)")


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
# marathon wall share per solver
for ax, name in zip(axes[:2], list(marathon_results)[:2]):
    s = marathon_results[name]
    acc = accepted_ids(s)
    cats = ["extra_hard", "order5"]
    by_tier = {c: sum(1 for i in acc if source_by_id[i]["difficulty"] == c) for c in cats}
    ax.bar(cats, [by_tier[c] for c in cats], color=["#4C72B0", "#DD8452"])
    ax.set_title(f"{name} accepted by tier (total {s['score']})")
    ax.set_ylim(0, 85)
    for c in cats:
        ax.text(c, by_tier[c] + 1, str(by_tier[c]), ha="center")
# solo elapsed distribution
ax = axes[2]
for name in ["solo300", "solo6"]:
    done = solo_done[name]
    vals = sorted(r["elapsed_seconds"] for r in done.values())
    ax.plot(vals, label=name)
ax.set_title("SOLO structural_cache per-problem elapsed (s, sorted)")
ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
import itertools
runs = {r["run"]: r["accepted_ids"] for _, r in df.iterrows()}
names = list(runs)
overlap = pd.DataFrame(index=names, columns=names, dtype=int)
for a, b in itertools.product(names, repeat=2):
    overlap.loc[a, b] = len(runs[a] & runs[b])
display(overlap)
print("\nExclusive wins per run:")
for a in names:
    exclusive = runs[a] - set().union(*(runs[b] for b in names if b != a)) if len(names) > 1 else runs[a]
    print(f"  {a}: {len(exclusive)}")
    for i in sorted(exclusive):
        print(f"     {i}  {source_by_id[i]['difficulty']}  {expected[i]}")


In [ ]:
def simulate_global_budget(done, budget=300.0, order=None):
    items = list(done.values())
    if order is not None:
        pos = {pid: k for k, pid in enumerate(order)}
        items.sort(key=lambda r: pos.get(r["id"], 10**9))
    elapsed = 0.0
    solved = 0
    for r in items:
        if elapsed + r["elapsed_seconds"] > budget:
            break
        elapsed += r["elapsed_seconds"]
        if r.get("label_match"):
            solved += 1
    return solved, elapsed

print("Greedy global-budget simulation for the SOLO solver (per-problem runs, naive port):")
manifest_order = [p["id"] for p in problems]
for name in ["solo300", "solo6"]:
    done = solo_done[name]
    s_manifest, e_manifest = simulate_global_budget(done, 300.0, manifest_order)
    # V3's priority order: reuse solverV3's problem_priority
    try:
        from my_submission.marathon import solverV3 as v3
        prio = []
        for p in problems:
            try:
                eq1 = v3.parse_equation(str(p["equation1"]))
                eq2 = v3.parse_equation(str(p["equation2"]))
                prio.append((v3.problem_priority(p, eq1, eq2), p["id"]))
            except Exception:
                prio.append(((9, 0, "skip"), p["id"]))
        prio.sort(key=lambda t: t[0])
        v3_order = [pid for _, pid in prio]
        s_v3, e_v3 = simulate_global_budget(done, 300.0, v3_order)
    except Exception as exc:
        print(f"  (V3 priority import failed: {exc})")
        s_v3, e_v3 = None, None


## 7 — Portability verdict (structural_cache for the marathon track)

Summarize after the runs complete. Reference expectations from the local benchmark (doc `docs/marathon_hard100_benchmark.md`):

- The marathon port **ties V3 at 90/100** (296.6s vs 147.1s) with complementary strengths: the port proves 8 true order-5 implications V3 misses (standard-lemma superposition), V3 wins 7 false counterexamples the port's row budget starves.
- Merging both accepted sets reaches **98/100**; only `evaluation_order5_0006` and `evaluation_order5_0124` defeat both.
- The SOLO runs will show how much per-problem budget the SOLO solver actually needs, and the 6s pass will show the degradation under marathon pressure.
- Expected conclusion: suitable in the narrow sense (it already ties the best marathon solver on the hardest slice with zero core changes); the porting cost is an adapter + row deadline, not solver surgery. The main risks are wall-time margin (2x V3) and mis-sequenced counterexample routes under per-row deadlines.


In [ ]:
# one-shot summary printer (fills the verdict cells above)
print("=== FINAL SCOREBOARD ===")
for _, r in df.iterrows():
    print(f"{r['run']:34s} {r['accepted']:3d}/100  F={r['false']:2d} T={r['true']:2d}  wall={r['wall_seconds']:.1f}s  budget={r['budget']}")
if "solverV3" in set(df["run"]):
    v3acc = df.loc[df["run"] == "solverV3", "accepted_ids"].iloc[0]
    port = df[df["run"].str.contains("structural_cache_marathon")]
    if not port.empty:
        pacc = port["accepted_ids"].iloc[0]
        print(f"\nunion(V3, port) = {len(v3acc | pacc)}/100")
        print(f"intersection = {len(v3acc & pacc)}")
        solo300 = df[df["run"].str.contains("solo300")]
        if not solo300.empty:
            sacc = solo300["accepted_ids"].iloc[0]
            print(f"solo300 exclusive vs marathon runs: {len(sacc - v3acc - pacc)}")
else:
    print("solverV3 missing from results")


## 8 — Export

Results are under `pipeline/results/marathon_hard100/` (marathon) and `pipeline/results/marathon_vs_solo/` (solo), mirrored to `/kaggle/working`.


In [ ]:
import shutil, pathlib
from pathlib import Path
import pandas as pd
if Path("/kaggle/working").exists():
    !mkdir -p /kaggle/working/results
    !cp -r pipeline/results/marathon_hard100 /kaggle/working/results/ 2>&1 | tail -2
    !cp -r pipeline/results/marathon_vs_solo /kaggle/working/results/ 2>&1 | tail -2
    print("Mirrored to /kaggle/working/results")
# combined per-problem CSV
allrows = []
for _, r in df.iterrows():
    for pid in r["accepted_ids"]:
        allrows.append({"run": r["run"], "id": pid,
                       "difficulty": source_by_id[pid]["difficulty"],
                       "expected": expected[pid]})
df_all = pd.DataFrame(allrows)
df_all.to_csv("marathon_v3_vs_structural_cache.csv", index=False)
print(f"wrote marathon_v3_vs_structural_cache.csv rows={len(df_all)}")
df_all.head()
